In [ ]:
from pyspark.sql import SparkSession
from minio import Minio
from pathlib import Path

In [ ]:
DIMENSOES = [
    "usuarios",
    "planos",
    "artistas",
    "albuns",
    "musicas",
    "generos",
    "playlists",
    "dispositivos",
]

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Detecta se está rodando localmente (VS Code) ou dentro do Docker
is_docker = os.path.exists('/.dockerenv')

POSTGRES_HOST = os.getenv("POSTGRES_HOST", "localhost") if is_docker else "localhost"
POSTGRES_PORT = os.getenv("POSTGRES_PORT", "5432")
POSTGRES_DB   = os.getenv("POSTGRES_DB", "streaming")
POSTGRES_USER = os.getenv("POSTGRES_USER", "streaming")
POSTGRES_PASS = os.getenv("POSTGRES_PASSWORD", "streaming123")

MINIO_HOST = ("minio:9000" if is_docker else "localhost:9000")
MINIO_USER = os.getenv("MINIO_ROOT_USER", "minioadmin")
MINIO_PASS = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

spark = (
    SparkSession.builder
    .appName("landing-dimensoes")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
    .getOrCreate()
)

cliente_minio = Minio(
    MINIO_HOST,
    access_key=MINIO_USER,
    secret_key=MINIO_PASS,
    secure=False
)

In [ ]:
buckets = cliente_minio.list_buckets()
for bucket in buckets:
    print(f"Bucket encontrado: {bucket.name}")

In [ ]:
jdbc_url = f"jdbc:postgresql://{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"

propriedades = {
    "user": POSTGRES_USER,
    "password": POSTGRES_PASS,
    "driver": "org.postgresql.Driver"
}

for tabela in DIMENSOES:
    print(f"\n{'=' * 50}")
    print(f"Processando tabela: {tabela}")
    print(f"{'=' * 50}")

    df = spark.read.jdbc(url=jdbc_url, table=tabela, properties=propriedades)

    total = df.count()
    print(f"Total de registros: {total}")

    caminho_local = f"/tmp/landing/{tabela}"

    df.write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(caminho_local)

    csv_dir = Path(caminho_local)
    arquivo_csv = next(
        arquivo for arquivo in csv_dir.iterdir()
        if arquivo.name.endswith(".csv")
    )

    cliente_minio.fput_object(
        "landing",
        f"{tabela}/{tabela}.csv",
        str(arquivo_csv)
    )

    print(f"{tabela}.csv enviado para o MinIO!")

print("\nIngestão das dimensões concluída com sucesso!")

In [ ]:
spark.stop()